# First Title
## Second Title
- item 1
- item 2
- item 3
**abc**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("RFM_Raw.csv")

In [3]:
df.head()

,Date,InvoNo,CustomerKey,Amount
0,2018/01/01,JA1803995,A0620,411.400
1,2018/01/01,JA1805774,A1336,658.240
2,2018/01/01,JA1801197,A0786,1892.352
3,2018/01/02,JA1805224,A0937,411.400
4,2018/01/02,JA1802973,A1371,411.400


In [5]:
import pandas as pd

# 假設你的原始 DataFrame 叫做 df
# df = pd.DataFrame(...)

# 1. 確保Date欄位為 datetime 型態，以便進行Date運算
df['Date'] = pd.to_datetime(df['Date'])

# 2. 找出整個資料集中的最大Date（作為基準日）
# 如果你是要用「今天」作為基準，可以改用 pd.Timestamp.today()
max_date = df['Date'].max()

# 3. 進行 groupby 並計算 F、M、R
df_rfm = (
    df.groupby('CustomerKey')
    .agg(
        # F: InvoNo的計數（計算交易次數）
        F=('InvoNo', 'count'),
        # M: Amount的加總
        M=('Amount', 'sum'),
        # R: 先找出該客戶的最大Date (last date)，後面再計算差額
        last_date=('Date', 'max'),
    )
    .reset_index()
)

# 4. 計算 R：基準最大Date 與 該客戶最大交易Date 的差額（單位：天數）
df_rfm['R'] = (max_date - df_rfm['last_date']).dt.days

# 5. 重新命名並調整欄位順序，符合你要求的四大欄位：Customer, F, M, R
df_rfm = df_rfm.rename(columns={'CustomerKey': 'Customer'})[
    ['Customer', 'F', 'M', 'R']
]

# 檢視結果
print(df_rfm)

    Customer   F            M   R
0      A0350  25  51238.11440   9
1      A0351   6  17890.31200  16
2      A0352  14  46424.03852  14
3      A0353  14  27607.68636  24
4      A0354   6   9973.92064  31
..       ...  ..          ...  ..
727    A1413  23  52738.79456  10
728    A1415  20  53019.49664  17
729    A1416  24  49938.38378   4
730    A1419  22  64257.84520  50
731    A1420  14  25054.18342  85

[732 rows x 4 columns]


In [8]:
df_rfm.to_csv("RFM_Data.csv",index=False)